## Simple object detection in TensorFlow

This notebook includes the following - 
1. Download the `Faster R-CNN` model from TensorFlow hub
2. Get an image from the internet and preprocess it using the model 
3. Use the imported model to get predictions on the image

In [1]:
!pip install tensorflow_hub

In [2]:
!pip install protobuf==4.21.6
!pip install tf-keras==2.19.0

In [3]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import tempfile
from PIL import Image, ImageOps
from six import BytesIO
from six.moves.urllib.request import urlopen

/Users/opmule/miniforge3/envs/ai_tensorflow/lib/python3.10/site-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


###

### Download & load the model from TensorFlow hub

This is the repository where trained ML models and neural networks are stored which we can reuse for our own projects<br>
For this notebook - We will be using: Image object detection & from this category the Inception resnet v2 model

In [6]:
model = hub.load("https://www.kaggle.com/models/tensorflow/faster-rcnn-inception-resnet-v2/TensorFlow2/640x640/1")

#### Signatures

Some models from tensorflow hub are used for multiple tasks so the documentation should mention which *signature* should be used for Object detection

In [12]:
# List all the signatures for the above model
print(list(model.signatures.keys()))

# Select default signature since we only have one here
detector = model.signatures['serving_default']

['serving_default']


### Download & resize image

Downloads an image using the given URL, pre-process it & saves it to disk

In [13]:
def download_resize_image(url, new_width=256, new_height=256):
    # Create temporary file (.jpg) & get the imagedata
    _, filename = tempfile.mkstemp(".jpg")
    response = urlopen(url)
    image_data = response.read()

    # Put image in memory buffer
    image_data = BytesIO(image_data)

    # Open image, resize, crop, convert to RGB & save to file
    pil_image = Image.open(image_data)
    pil_image = ImageOps.fit(pil_image, (new_width, new_height), Image.Resampling.LANCZOS)
    pil_image_rgb = pil_image.convert("RGB")
    pil_image_rgb.save(filename, format="JPEG", quality=90)
    print(f"Image downloaded to {filename}")
    return filename

In [17]:
image_url = "https://i.pinimg.com/736x/12/f7/34/12f7344b2377668026ef065b79f24eda.jpg"
downloaded_image_path = download_resize_image(image_url, 640, 640)

Image downloaded to /var/folders/f2/j66wx7px7sx9qrws63ymk_sr0000gn/T/tmp7tgqe7m2.jpg


### Get predictions using loaded object detection model

Runs the detector model on the sample downloaded image and gets the detected objects & bounding boxes

In [23]:
def load_img(path):
    img = tf.io.read_file(path)
    # Convert image to tensor
    img = tf.image.decode_jpeg(img, channels=3)
    return img

def run_detector(detector, path):
    img = load_img(path)
    # Add a batch dimension in front of the tensor
    # tf.newaxis adds a new dimension to the start of the image dimensions & ... means
    # keeping the rest as it is (256,256,3) -> (1,256,256,3)
    converted_img = tf.image.convert_image_dtype(img, tf.uint8)[tf.newaxis, ...]
    result = detector(converted_img)
    # Save results to dictionary
    result = {key:value.numpy() for key,value in result.items()}
    # print results
    print("Found %d objects." % len(result["detection_scores"]))
    print(result["detection_scores"])
    print(result["detection_classes"])  # Changed from detection_class_entities
    print(result["detection_boxes"])

### Perform object detection on the saved image

In [24]:
run_detector(detector, downloaded_image_path)

Found 1 objects.
[[0.9773781  0.93808013 0.8644066  0.81958383 0.8131412  0.78454554
  0.7108988  0.69759816 0.696581   0.6678349  0.47073898 0.42270878
  0.4130443  0.32111216 0.31654945 0.31300005 0.30779317 0.28020057
  0.26941597 0.26127478 0.25867185 0.24038649 0.23856673 0.23270346
  0.23236477 0.23212312 0.22834074 0.22289571 0.22140068 0.20072395
  0.18773042 0.18387808 0.1831883  0.18035091 0.1710275  0.16109195
  0.15989971 0.15619372 0.15494539 0.15460719 0.15355812 0.15142407
  0.14843409 0.14421311 0.14141478 0.12830684 0.12403663 0.10144013
  0.09652519 0.09550828 0.09315872 0.09259357 0.08976413 0.08800529
  0.08169393 0.07957409 0.07731658 0.07155921 0.07125191 0.0685778
  0.06754429 0.06637967 0.06506021 0.06368025 0.06250686 0.06084695
  0.06005573 0.05870492 0.05768994 0.05722154 0.05630239 0.05481054
  0.05067237 0.04998431 0.04932066 0.04853445 0.04850781 0.04829495
  0.04787061 0.0456617  0.04394461 0.04365313 0.04288208 0.0383341
  0.03823878 0.03715822 0.0369909